# Step 5 — Engineer useful features

This notebook creates only the practical features needed for the three analysis
questions:

- Company age
- Funding delay and funding span
- Funding bands
- Failure-factor counts and risk combinations
- Runway bands
- Peer cohorts

The three datasets remain separate. The Step 4 tables are read from
`data/processed`, and new feature tables are saved back to that folder.


## Feature definitions

**Crunchbase features**

- `company_age_years`: years from founding to the dataset reference date.
- `funding_delay_months`: months from founding to first funding.
- `funding_span_months`: months from first funding to last funding.
- `funding_band`: a simple business-friendly funding range.
- `founded_cohort`: a five-year founding-period group.
- `peer_cohort`: primary category + country + founding cohort.

**Failure features**

- `failure_factor_count`: number of recorded failure flags equal to 1.
- `risk_combination`: readable list of the recorded failure factors.
- `operating_years`: end year minus start year.
- `peer_cohort`: sector + funding band.

**Startup-metrics features**

- `runway_band`: practical cash-runway group.
- `peer_cohort`: stage + category group.

Each peer cohort also receives a `peer_cohort_size`. Small cohorts should be
interpreted cautiously in later analysis.


In [ ]:
# Import the libraries used in this notebook.
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        "The data/processed folder was not found. "
        "Run the Step 4 notebook first."
    )

FAILURE_FILE = PROCESSED_DIR / "failure_analysis_table.csv"
CRUNCHBASE_FILE = PROCESSED_DIR / "crunchbase_analysis_table.csv"
METRICS_FILE = PROCESSED_DIR / "startup_metrics_analysis_table.csv"

required_files = [FAILURE_FILE, CRUNCHBASE_FILE, METRICS_FILE]
missing_files = [str(file) for file in required_files if not file.exists()]

if missing_files:
    raise FileNotFoundError(f"Missing Step 4 files: {missing_files}")

print("All Step 4 analysis tables were found.")


## 1. Define simple business bands

Fixed bands are easier to explain and reuse than bands that change whenever the
data change.

Funding bands:

- Under $1M
- $1M to under $5M
- $5M to under $20M
- $20M to under $100M
- $100M or more

Runway bands:

- Under 3 months
- 3 to under 6 months
- 6 to under 12 months
- 12 to under 18 months
- 18 months or more


In [ ]:
def create_funding_band(values):
    # Group funding totals into readable ranges.
    bands = pd.cut(
        values,
        bins=[-float("inf"), 1_000_000, 5_000_000, 20_000_000, 100_000_000, float("inf")],
        labels=[
            "Under $1M",
            "$1M to under $5M",
            "$5M to under $20M",
            "$20M to under $100M",
            "$100M or more",
        ],
        right=False,
    )
    return bands.astype("string").fillna("Missing")


def create_founded_cohort(years):
    # Group founding years into five-year periods.
    cohorts = pd.cut(
        years,
        bins=[-float("inf"), 2000, 2005, 2010, 2015, float("inf")],
        labels=[
            "Before 2000",
            "2000-2004",
            "2005-2009",
            "2010-2014",
            "2015 or later",
        ],
        right=False,
    )
    return cohorts.astype("string").fillna("Unknown")


## 2. Create Crunchbase features

The dataset does not provide an exact observation date. We use December 31 of
the latest first-funding year as a transparent reference date. In this dataset,
that year is 2015.

`company_age_years` is therefore age at the dataset reference date, not confirmed
operating lifespan. Dates after the reference date and first-funding dates before
founding are flagged and excluded from the affected calculations.


In [ ]:
date_columns = ["founded_at", "first_funding_at", "last_funding_at"]
crunchbase = pd.read_csv(
    CRUNCHBASE_FILE,
    parse_dates=date_columns,
    low_memory=False,
)

# Use the latest first-funding year to define the dataset reference year.
reference_year = int(crunchbase["first_funding_at"].dt.year.max())
reference_date = pd.Timestamp(year=reference_year, month=12, day=31)

# Flag suspicious dates instead of silently deleting their rows.
crunchbase["date_after_reference"] = (
    (crunchbase["founded_at"] > reference_date)
    | (crunchbase["first_funding_at"] > reference_date)
    | (crunchbase["last_funding_at"] > reference_date)
)
crunchbase["funding_before_founding"] = (
    crunchbase["first_funding_at"] < crunchbase["founded_at"]
)

# Company age at the reference date.
crunchbase["company_age_years"] = (
    (reference_date - crunchbase["founded_at"]).dt.days / 365.25
).round(1)
crunchbase.loc[
    crunchbase["founded_at"] > reference_date,
    "company_age_years",
] = pd.NA

# Months from founding to first funding.
crunchbase["funding_delay_months"] = (
    (crunchbase["first_funding_at"] - crunchbase["founded_at"]).dt.days / 30.44
).round(1)
crunchbase.loc[
    crunchbase["funding_before_founding"]
    | (crunchbase["first_funding_at"] > reference_date),
    "funding_delay_months",
] = pd.NA

# Months between first and last funding.
crunchbase["funding_span_months"] = (
    (crunchbase["last_funding_at"] - crunchbase["first_funding_at"]).dt.days / 30.44
).round(1)
crunchbase.loc[
    (crunchbase["funding_span_months"] < 0)
    | (crunchbase["last_funding_at"] > reference_date),
    "funding_span_months",
] = pd.NA

# Funding and founding-period bands.
crunchbase["funding_band"] = create_funding_band(crunchbase["funding_total_usd"])
crunchbase["founded_year"] = crunchbase["founded_at"].dt.year.astype("Int64")
crunchbase["founded_cohort"] = create_founded_cohort(crunchbase["founded_year"])

# Comparable companies share category, country, and founding cohort.
crunchbase["peer_cohort"] = (
    crunchbase["primary_category"].astype("string")
    + " | " + crunchbase["country_code"].astype("string")
    + " | " + crunchbase["founded_cohort"].astype("string")
)
crunchbase["peer_cohort_size"] = (
    crunchbase.groupby("peer_cohort")["permalink"].transform("size")
)

CRUNCHBASE_OUTPUT = PROCESSED_DIR / "crunchbase_feature_table.csv"
crunchbase.to_csv(CRUNCHBASE_OUTPUT, index=False, date_format="%Y-%m-%d")

print("Reference date:", reference_date.date())
print("Rows:", len(crunchbase))
print("Dates after the reference date:", crunchbase["date_after_reference"].sum())
print("First funding before founding:", crunchbase["funding_before_founding"].sum())
print("Missing company age:", crunchbase["company_age_years"].isna().sum())
print("Saved:", CRUNCHBASE_OUTPUT)


## 3. Create failure features

The factor count includes only flags equal to 1. Missing flags are ignored and
are not treated as evidence that a factor was absent.

`risk_combination` is a readable description of the active factors for each
failed startup. It is descriptive; it does not establish which factor caused the
failure.


In [ ]:
failure = pd.read_csv(FAILURE_FILE, low_memory=False)

failure_flags = [
    "Giants", "No Budget", "Competition", "Poor Market Fit",
    "Acquisition Stagnation", "Platform Dependency",
    "High Operational Costs", "Monetization Failure", "Niche Limits",
    "Execution Flaws", "Trend Shifts", "Toxicity/Trust Issues",
    "Regulatory Pressure", "Overhype",
]

# Count active failure factors.
failure["failure_factor_count"] = failure[failure_flags].eq(1).sum(axis=1)


def list_active_risks(row):
    # Return a readable list of flags equal to 1.
    active = [flag for flag in failure_flags if row[flag] == 1]
    return " + ".join(active) if active else "No recorded factor"


failure["risk_combination"] = failure.apply(list_active_risks, axis=1)

# Operating duration from the parsed start and end years.
failure["operating_years"] = failure["end_year"] - failure["start_year"]
failure.loc[failure["operating_years"] < 0, "operating_years"] = pd.NA

# Use the same funding bands as the Crunchbase table.
failure["funding_band"] = create_funding_band(failure["funding_raised_usd"])

# Failure peers share sector and funding band.
failure["peer_cohort"] = (
    failure["Sector"].astype("string")
    + " | " + failure["funding_band"].astype("string")
)
failure["peer_cohort_size"] = (
    failure.groupby("peer_cohort")["failure_record_id"].transform("size")
)

FAILURE_OUTPUT = PROCESSED_DIR / "failure_feature_table.csv"
failure.to_csv(FAILURE_OUTPUT, index=False)

print("Rows:", len(failure))
print("Average recorded factors:", round(failure["failure_factor_count"].mean(), 2))
print("Unique risk combinations:", failure["risk_combination"].nunique())
print("Saved:", FAILURE_OUTPUT)


## 4. Create startup-metrics features

Runway bands provide a simple view of immediate cash risk. The thresholds are
practical categories for analysis, not universal investment rules.

Metrics peers share the same stage and category group.


In [ ]:
metrics = pd.read_csv(METRICS_FILE)

# Create practical runway groups.
metrics["runway_band"] = pd.cut(
    metrics["runway_months"],
    bins=[-float("inf"), 3, 6, 12, 18, float("inf")],
    labels=[
        "Under 3 months",
        "3 to under 6 months",
        "6 to under 12 months",
        "12 to under 18 months",
        "18 months or more",
    ],
    right=False,
).astype("string").fillna("Missing")

# Metrics peers share stage and category group.
metrics["peer_cohort"] = (
    metrics["stage"].astype("string")
    + " | " + metrics["category_group"].astype("string")
)
metrics["peer_cohort_size"] = (
    metrics.groupby("peer_cohort")["startup_id"].transform("size")
)

METRICS_OUTPUT = PROCESSED_DIR / "startup_metrics_feature_table.csv"
metrics.to_csv(METRICS_OUTPUT, index=False)

print("Rows:", len(metrics))
print("Runway-band counts:")
print(metrics["runway_band"].value_counts().to_string())
print("Saved:", METRICS_OUTPUT)


## 5. Check and summarize the feature tables

This final check confirms that feature engineering did not change the number of
records in any main table.


In [ ]:
feature_summary = pd.DataFrame(
    [
        {
            "table": CRUNCHBASE_OUTPUT.name,
            "rows": len(crunchbase),
            "columns": len(crunchbase.columns),
            "new_features": (
                "company age, funding delay, funding span, funding band, "
                "founding cohort, peer cohort"
            ),
        },
        {
            "table": FAILURE_OUTPUT.name,
            "rows": len(failure),
            "columns": len(failure.columns),
            "new_features": (
                "failure-factor count, risk combination, operating years, "
                "funding band, peer cohort"
            ),
        },
        {
            "table": METRICS_OUTPUT.name,
            "rows": len(metrics),
            "columns": len(metrics.columns),
            "new_features": "runway band, peer cohort",
        },
    ]
)

SUMMARY_OUTPUT = PROCESSED_DIR / "feature_table_summary.csv"
feature_summary.to_csv(SUMMARY_OUTPUT, index=False)

print(feature_summary.to_string(index=False))
print("\nSaved summary:", SUMMARY_OUTPUT)


## Step 5 result

The following feature tables are ready for exploratory analysis:

- `crunchbase_feature_table.csv`
- `failure_feature_table.csv`
- `startup_metrics_feature_table.csv`

Important interpretation notes:

- Company age is measured at the dataset reference date, not at closure or exit.
- Missing funding and dates remain missing; they are not replaced with zero.
- Risk combinations describe recorded factors and do not prove causation.
- Peer-cohort size must be checked before drawing conclusions from a cohort.

No statistical modelling or exploratory charts are performed in this notebook.
